In [ ]:
import os
import sys
import pathlib

# Set to the root directory - 
# only allow this to run once in the jupyter session
if 'changed_root' not in locals():
    os.chdir(os.path.dirname(os.getcwd()))
    changed_root = True
print(os.getcwd())


# Brainwear Pipeline

Replicates the CIFAR-10 pretrain -> AACBR -> contest -> global-optimize pipeline on wearable accelerometer data (one mock patient, `data/brainwear/BW_AAB_mastertimeSeries.csv`).

**Task framing:** the raw data is one continuous year-long series of 30-second epochs (mean acceleration, an `imputed` flag, and an activity label from a classifier trained on healthy patients: sleeping/sedentary/light-tasks/moderate/walking). A single epoch carries no learnable structure, so cases are **10-minute windows** (20 consecutive epochs) instead -- the temporal analogue of a CIFAR-10 image. Each window's target is the majority-vote activity label across its 20 epochs.

Designed to survive more patients being added later: swapping in a multi-patient dataset only touches the preprocessing step (windowing/labeling logic is per-recording already), not the model configs downstream.

## Preprocessing

Raw CSV (one row per 30s epoch) -> windowed, tabular train/val/test CSVs the existing `tabular` data loader reads unmodified.

- **Window = 20 epochs = 10 minutes**, non-overlapping (stride = window length). ~40k windows survive from ~1M epochs -- comparable in scale to CIFAR-10's 60k images, so there's no need to manufacture more via overlapping windows, and non-overlap means no near-duplicate windows to leak across a split.
- **Windows never cross a real time gap** in the recording (device off/removed) -- there are 7 contiguous runs across the year.
- **Windows containing any NaN acceleration reading are dropped** -- the raw `acceleration(mg)` column has ~3,864 genuinely missing (not just interpolated) values, separate from the `imputed` flag; a NaN can't be fed to the model, so this is a hard filter, not a tunable quality threshold like the two below.
- **Windows with >20% interpolated epochs are dropped** (`imputed` used as a data-quality filter, not a model feature).
- **Split is chronological, not random**: earliest 80% of windows (by time) -> `brainwear_trainval.csv`, latest 20% -> `brainwear_test.csv`. The existing tabular loader's own `path`/`test_path` mechanism then carves val out of trainval via a random 80/20 split (~64%/16%/20% overall) -- val isn't a separate chronological block since the loader has no third split hook, but since windows are non-overlapping this only risks mild temporal-autocorrelation between adjacent windows, not literal duplicate-row leakage. Test is genuinely held out at the end of the year -- a real "generalize to the future" check, and immune to any overlap/stride choice.
- **No cross-validation** for this pass: k-fold would randomly mix time periods and reintroduce the leakage the chronological split is avoiding; the CLI's own multi-seed repeats (`--seed "(0, 5)"`) already average out init/training variance. The CV that will actually matter for this project is leave-one-patient-out once more patients exist -- revisit then.
- `MET` and the raw per-epoch activity columns are used only to build the target label, then dropped -- `MET` is a near-deterministic function of the activity label (confirmed empirically), so keeping it as an input would leak the label.

In [ ]:
! python -m scripts.preprocess_brainwear --window-epochs 20 --test-frac 0.2

In [ ]:
import json

from IPython.display import Image, display

from deeparguing.output_paths import find_output

with open("data/brainwear/window_meta.json") as f:
    manifest = json.load(f)
display(manifest)

display(Image(filename=find_output("brainwear_window_label_distribution.png")))

## Pretrain the embedding

Plays `ResNetCIFAR`'s role: pretrain a sequence embedding, then freeze it for the AACBR model. `LSTMFeatureExtractor` (already in the repo, previously unused by any tuning config) takes a length-20 acceleration sequence and compresses it to a 32-dim embedding.

Since `LSTMFeatureExtractor` has no built-in classification head (unlike `ResNetCIFAR`'s `use_classification_head`), `pretrain_lstm.py` bolts on a throwaway `nn.Linear(32, 5)` head, trains both jointly as a plain activity classifier on `brainwear_trainval.csv`, then saves only the LSTM's weights -- the embedding is deliberately wider than the 5-class output so it survives pretraining as a real representation rather than collapsing to logits.

Loads via the same `load_tabular_data` + `train_test_split(test_size=0.2, random_state=42)` calls the downstream AACBR run's `tabular`+`test_path` data loader uses, so the embedding is never pretrained on a different train/val split or a different feature scaling than what it will see frozen inside the AACBR model.

**Class imbalance:** `sleep`/`sedentary` dominate the window counts (see the preprocessing distribution above) -- training uses inverse-frequency class weighting in the cross-entropy loss, or the model just learns to always predict the majority classes.

In [ ]:
! python -m scripts.pretrain_lstm --epochs 20 --hidden-size 64 --embedding-size 32

In [ ]:
from IPython.display import Image, display

from deeparguing.output_paths import find_output

display(Image(filename=find_output("lstm_brainwear_per_class_f1.png")))
print("Checkpoint:", find_output("lstm_brainwear.pt"))

## Deep Arguing Model

`tuning/brainwear/{data,hyperparameters,model}_brainwear.yaml` -- mirrors `tuning/cifar10/resnet/relu` structurally (same regularisers/trainer/casebase/semantics wiring), swapping in what's dataset-specific:

- **`feature_weights`**: `LSTMFeatureExtractor` (hidden_size=64 -> embedding=32) loading `lstm_brainwear.pt`, frozen -- plays `ResNetCIFAR`'s role exactly. Architecture here must match `pretrain_lstm.py`'s `--hidden-size`/`--embedding-size`/`--num-layers` exactly, since `weights_path` loads a `state_dict` into this exact shape.
- **`feature_weights_1`/`feature_weights_2`**: the same two small `MLPExtractor` heads CIFAR uses (comparison embedding + base score), input size dropped from 64 to 32 to match the LSTM's embedding width.
- **`semantics`**: `ReluSemantics`, as requested -- CIFAR moved off this to `QuadraticEnergySemantics` after `contest`/`global_optimize` made target-class strengths blow up unboundedly (see `updates.md`, 2026-07-14). Untested here yet; worth watching for the same failure mode once we reach the contest stage.
- **`build_casebase`**: `kMeansCluster(cluster_size=5)`, same flat per-class count as CIFAR (5 clusters/class x 5 classes = 25 casebase nodes). Numerically fine even for the smallest classes (moderate/tasks-light have hundreds of training windows, cluster_size=5 is far below that) -- but see the smoke-test result below.
- **Data**: reuses `brainwear_trainval.csv`/`brainwear_test.csv` via the plain `tabular` loader (`path`/`test_path`, `target_field=20`, `continuous_cols=range(0,20)`) -- no new data-loading code.
- **Hyperparameters**: starting defaults copied from CIFAR's relu config, *not yet tuned for this dataset* -- no `tune_hyperparameters_brainwear.yaml`/optuna sweep has been run.

**Known gap:** the main AACBR training loss (`TorchLoss`/`CrossEntropyLoss`) has no class-weighting hook in the current YAML plumbing (`weight` needs a `Tensor`, and nothing in `parse_yaml.py` builds one from config) -- unlike `pretrain_lstm.py`'s loss, which does weight by inverse frequency. A single-seed smoke test of this exact config found the model never predicts `tasks-light` at all (0 predictions across train/val/test, despite ~145-830 true examples in each split) -- test accuracy 74.4% / macro F1 0.43. The pipeline runs end-to-end without errors, but this is a real quality gap worth revisiting (candidates: per-class `cluster_size` as a list instead of one flat int so the casebase gives `tasks-light` proportionally more representation, or adding a weighted-loss mechanism to `parse_yaml.py`) rather than something silently fixed here.

`--misclassified_log --grae_log` export the misclassified test samples and their G-RAE attributions the next stage (`contest`) needs.

In [ ]:
! python src/deeparguing/cli/run.py --config "tuning/brainwear/data_brainwear.yaml" "tuning/brainwear/hyperparameters_brainwear.yaml" "tuning/brainwear/model_brainwear.yaml" --seed "(0, 5)" --log "info" --run_train -lv --run_test --misclassified_log --grae_log

In [ ]:
from IPython.display import Markdown, display

from deeparguing.output_paths import find_output

with open(find_output("summary.md")) as f:
    display(Markdown(f.read()))

## Contest

`contest_all.py` runs a single joint gradient-guided edit over every misclassified test sample against the shared casebase adjacency (`model.A`), via `batch_contest()`. Fully dataset-agnostic -- reads `model_checkpoint.pt`/`misclassified_qbaf.json` (bare filenames, auto-resolved to the most recent dated `outputs/` folder that has them) and the existing `tuning/contest/contest.yaml` config, unchanged from CIFAR's. No new brainwear-specific file needed for this stage.

**This has no accuracy protection** -- it only tries to flip misclassified samples, with no guardrail on what that does to everything else. A single-seed smoke test of this exact setup: cleared 197/2077 samples (9.5%), touching only 3 edges, converging in 2 iterations -- sounds contained. It was not: global test accuracy collapsed **74.4% -> 9.0%** (worse than random guessing on 5 classes). Confirmed by `run_global_contest_eval.py` below.

This is the same failure mode CIFAR hit with `ReluSemantics` (see `updates.md`, 2026-07-14: unbounded strength propagation means a handful of edited edges can cascade through the argumentation graph's fixed-point iteration and corrupt predictions for samples that never went near the edited edges) -- evidently worse here, likely because this casebase is much smaller (25 nodes vs. CIFAR's 50), so the same 3 edges are a much larger fraction of the graph's structure.

In [ ]:
! python -m deeparguing.contest.contest_all

In [ ]:
! python -m deeparguing.contest.run_global_contest_eval

## Global Optimize

`global_optimize.py` wraps `batch_contest()` in a guarded loop, built for exactly the failure just shown: a soft per-step penalty (`protect_lambda`) that discourages moving a sampled set of currently-correct held-out points, plus a hard guardrail that re-checks real validation accuracy every `eval_every` iterations and **rolls back to the last passing snapshot** if it drops more than `max_acc_drop`.

Result on this checkpoint: **0/2077 cleared, 0 edges changed, accuracy held exactly at baseline (0.7552), `rolled_back=True`, `stopped_reason=acc_drop`.** The guardrail worked correctly -- it caught the same instability `contest_all` just showed and reverted before any damage landed. But the practical outcome is that **no net benefit is currently achievable** under `ReluSemantics` with `tuning/contest/global_optimize.yaml`'s current hyperparameters (`protect_lambda=50`, `eval_every=5`, `max_acc_drop=0.01`) -- those were tuned for CIFAR's `QuadraticEnergySemantics` checkpoint, a fundamentally more stable (bounded-strength) setting, not for `ReluSemantics`'s unbounded dynamics on this casebase.

In [ ]:
! python -m deeparguing.contest.global_optimize

In [ ]:
from IPython.display import Markdown, display

from deeparguing.output_paths import find_output

with open(find_output("global_contest_eval.md")) as f:
    display(Markdown(f.read()))
with open(find_output("global_optimize.md")) as f:
    display(Markdown(f.read()))

## Sweeping global_optimize for ReluSemantics

`global_optimize.yaml`'s current hyperparameters (`protect_lambda=50`, `eval_every=5`, `max_acc_drop=0.01`) were tuned for CIFAR's `QuadraticEnergySemantics` checkpoint, not for `ReluSemantics`. `sweep_global_optimize.py` grid-sweeps the five knobs it controls (`protect_lambda`, `eval_every`, `protect_margin`, `protect_sample_size`, `max_acc_drop`) against this checkpoint, mirroring exactly what CIFAR's own history did after hitting the same wall.

**Result: 0/32 combos cleared a single sample. 100% rollback rate. The marginal-effect table shows literally none of the five swept knobs move the outcome at all** -- every value of every hyperparameter, including the tightest possible check interval (`eval_every=1`), produces an identical 0% success / 100% rollback.

That result changes the diagnosis: `eval_every=1` means the guardrail checks after *every single iteration*, so if even that still rolls back every time, the very first iteration alone is already blowing the accuracy budget -- this isn't a "check more often" problem, the outer guardrail knobs can't fix it. The knob that actually controls how much damage one iteration does (`batch_contest`'s own `alpha_init`/`k`/`divergence_bound`) isn't part of this sweep. Direct test below.

In [ ]:
! python -m deeparguing.contest.sweep_global_optimize

In [ ]:
from IPython.display import Markdown, display

from deeparguing.output_paths import find_output

with open(find_output("global_optimize_sweep.md")) as f:
    display(Markdown(f.read()))

### Isolating step size from check frequency

A direct test: contest with a single, near-negligible step (`alpha_init=0.001` instead of the default `1.0`, `k=1` instead of `3`, `divergence_bound=2.0` instead of `100.0`, `max_iters=1`) -- as conservative an edit as the algorithm can make.

**Result: cleared 59/2077 samples (2.8%) by moving exactly 1 edge -- and global test accuracy still collapsed 74.4% -> 28.8% (-45.6 points).** Even the smallest possible unit of progress is catastrophic. This confirms it's not a step-size or check-frequency problem at all: with only 25 casebase nodes, a single edited edge apparently carries enough structural weight in `ReluSemantics`'s unbounded fixed-point iteration to corrupt predictions for thousands of samples that never touched that edge. No combination of `contest`/`global_optimize` hyperparameters can fix this -- it's the semantics/casebase-size combination itself.

In [ ]:
! python -m deeparguing.contest.contest_all --max-iters 1 --alpha-init 0.001 --k 1 --divergence-bound 2.0 --checkpoint-filename contested_tiny_step.pt --log-prefix contestation_tiny

In [ ]:
! python -m deeparguing.contest.run_global_contest_eval --contested-checkpoint contested_tiny_step.pt --log-path ""